<a href="https://colab.research.google.com/github/diegocortes30/pharma_batch_quality_analysis/blob/main/03_data_cleaning_pharma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Pharma Batch Quality Analysis: 03 Data Cleaning**

**Goal:** clean the three tables and merge them into a single reliable dataset for the analysis and the dashboard.

**Input:** the 3 original CSV files in `raw/` (never modified).

**Output:** 1 clean CSV file in `processed/`.

**Data:** [Big data collection in pharmaceutical manufacturing](https://figshare.com/collections/Cholessterol-Lowering_Drug_Process_and_Quality_Data/5645578) (figshare), 3 CSV files: Laboratory, Process, Normalization.

**Source:** Žagar, J. & Mihelič, J. (2022). Big data collection in pharmaceutical manufacturing and its use for product quality predictions. Scientific Data, 9(1), 99. https://doi.org/10.1038/s41597-022-01203-x

**Contents:** setup and data loading, table cleaning (Laboratory, Process, Normalization), missing values, table merge, final checks, save, cleaning summary.

## 1. Setup and data loading

In [353]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd

RAW = '/content/drive/MyDrive/pharma_batch_quality_analysis/data/raw'
PROCESSED = '/content/drive/MyDrive/pharma_batch_quality_analysis/data/processed'
os.makedirs(PROCESSED, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [354]:
laboratory = pd.read_csv('/content/drive/MyDrive/pharma_batch_quality_analysis/data/raw/Laboratory.csv', sep=';')
process = pd.read_csv('/content/drive/MyDrive/pharma_batch_quality_analysis/data/raw/Process.csv', sep=';')
normalization = pd.read_csv('/content/drive/MyDrive/pharma_batch_quality_analysis/data/raw/Normalization.csv', sep=';')

## 2. `Laboratory` cleaning

In [355]:
# Look at the shape
laboratory.shape

(1005, 55)

In [356]:
laboratory.head()

,batch,code,strength,size,start,api_code,api_batch,smcc_batch,lactose_batch,starch_batch,...,tbl_tensile,fct_tensile,tbl_yield,batch_yield,dissolution_av,dissolution_min,resodual_solvent,impurities_total,impurity_o,impurity_l
0,1,25,5MG,240000,nov.18,5,2,1,2,1,...,1.412698,1.926183,95.785,94.697,93.83,86,0.06,0.33,0.05,0.16
1,2,25,5MG,240000,nov.18,5,2,1,2,1,...,1.412698,1.986377,98.467,97.348,99.67,92,0.04,0.34,0.06,0.16
2,3,25,5MG,240000,nov.18,5,2,1,2,1,...,1.412698,2.016473,98.496,99.242,97.33,92,0.03,0.28,0.05,0.16
3,4,25,5MG,240000,nov.18,5,2,1,2,1,...,1.474120,1.956280,97.736,98.106,94.50,89,0.03,0.30,0.05,0.18
4,5,25,5MG,240000,nov.18,5,2,1,2,1,...,1.443409,1.926183,98.106,98.106,92.00,88,0.04,0.31,0.05,0.18


In [357]:
# Renaming column with typo
laboratory = laboratory.rename(columns={'resodual_solvent': 'residual_solvent'})
laboratory['residual_solvent'].head()

,residual_solvent
0,0.06
1,0.04
2,0.03
3,0.03
4,0.04


In [358]:
# Show only 'object' type columns
laboratory.select_dtypes(include=['object']).head()

,strength,start,api_water,api_total_impurities,api_l_impurity,api_ps01,api_ps05,api_ps09
0,5MG,nov.18,1.53,0.25,0.13,1.27,18.52,109.999
1,5MG,nov.18,1.53,0.25,0.13,1.27,18.52,109.999
2,5MG,nov.18,1.53,0.25,0.13,1.27,18.52,109.999
3,5MG,nov.18,1.53,0.25,0.13,1.27,18.52,109.999
4,5MG,nov.18,1.53,0.25,0.13,1.27,18.52,109.999


In [359]:
# Convert api columns to numerical
laboratory['api_water'] = pd.to_numeric(laboratory['api_water'], errors='coerce')
laboratory['api_total_impurities'] = pd.to_numeric(laboratory['api_total_impurities'], errors='coerce')
laboratory['api_l_impurity'] = pd.to_numeric(laboratory['api_l_impurity'], errors='coerce')
laboratory['api_ps01'] = pd.to_numeric(laboratory['api_ps01'], errors='coerce')
laboratory['api_ps05'] = pd.to_numeric(laboratory['api_ps05'], errors='coerce')
laboratory['api_ps09'] = pd.to_numeric(laboratory['api_ps09'], errors='coerce')

In [360]:
# Check if it works
laboratory[['api_water', 'api_total_impurities', 'api_l_impurity', 'api_ps01', 'api_ps05', 'api_ps09']].dtypes

,0
api_water,float64
api_total_impurities,float64
api_l_impurity,float64
api_ps01,float64
api_ps05,float64
api_ps09,float64


In [361]:
# Delete 'MG' and 'M' from strength column
laboratory['strength'] = laboratory['strength'].str.replace(r'[A-Za-z]', '', regex=True).astype(int)
laboratory['strength'].head()

,strength
0,5
1,5
2,5
3,5
4,5


In [362]:
# Check the start column
laboratory['start'].value_counts().sort_index()

,count
start,
apr.19,53
apr.20,4
apr.21,38
avg.19,23
avg.20,13
dec.18,9
dec.19,35
dec.20,12
feb.19,20


In [363]:
# Translate maj, avg, okt to english
laboratory['start'] = laboratory['start'].replace({'maj.19': 'may.19', 'maj.20': 'may.20',
                                                   'avg.19': 'aug.19', 'avg.20': 'aug.20', 'okt.19': 'oct.19', 'okt.20': 'oct.20'})
laboratory['start'].value_counts().sort_index()

,count
start,
apr.19,53
apr.20,4
apr.21,38
aug.19,23
aug.20,13
dec.18,9
dec.19,35
dec.20,12
feb.19,20


In [364]:
# Change start column to datetime
laboratory['start'] = pd.to_datetime(laboratory['start'], format='%b.%y')
laboratory['start'].head()

,start
0,2018-11-01
1,2018-11-01
2,2018-11-01
3,2018-11-01
4,2018-11-01


In [365]:
# Look for hardness columns
laboratory.filter(like='hardness').head()

,tbl_min_hardness,tbl_max_hardness,tbl_av_hardness,fct_min_hardness,fct_max_hardness,fct_av_hardness
0,56.84,68.60,46,37.0,56.0,62.72
1,56.84,70.56,46,39.0,56.0,64.68
2,58.80,70.56,46,39.0,57.0,65.66
3,58.80,72.52,48,57.0,40.0,63.70
4,56.84,68.60,47,39.0,59.0,62.72


In [366]:
# Delete hardness columns because the average hardness falls outside the min-max range; tensile strength will be used instead
laboratory = laboratory.drop(columns=['tbl_min_hardness', 'tbl_max_hardness', 'tbl_av_hardness', 'fct_min_hardness',
                         'fct_max_hardness', 'fct_av_hardness'])
laboratory.shape

(1005, 49)

In [367]:
# Fix batch size for codes 14 and 16 (4,800,000 -> 480,000, consistent with the normalization factor)
laboratory.loc[laboratory['code'].isin([14, 16]), 'size'] = 480000
laboratory['size'].value_counts()

,count
size,
960000,437
240000,218
1920000,211
1100000,76
480000,28
583000,22
1200000,8
2400000,5


In [368]:
# Check that batches follow chronological order
laboratory.sort_values('batch')['start'].is_monotonic_increasing

False

In [369]:
# Show batches whose start month is earlier than the previous batch
lab_sorted = laboratory.sort_values('batch')
lab_sorted[lab_sorted['start'] < lab_sorted['start'].shift()][['batch', 'code', 'start']]

,batch,code,start
61,73,23,2018-11-01


## 3. `Process` cleaning

In [370]:
# Look at the shape
process.shape

(1005, 35)

In [371]:
process.head()

,batch,code,tbl_speed_mean,tbl_speed_change,tbl_speed_0_duration,total_waste,startup_waste,weekend,fom_mean,fom_change,...,ejection_min,Startup_tbl_fill_maxDifference,Startup_main_CompForce_mean,Startup_tbl_fill_mean,Drug release average (%),Drug release min (%),Residual solvent,Total impurities,Impurity O,Impurity L
0,1,25,99.864656,5.416667,149.583333,2125.416667,5085,no,49.961446,12,...,196,0.38,4.587500,5.466667,93.83,86.0,0.06,0.33,0.05,0.16
1,2,25,99.936342,2.500000,128.333333,887.500000,2115,no,49.962040,5,...,194,0.18,4.390909,5.315455,99.67,92.0,0.04,0.34,0.06,0.16
2,3,25,99.985984,2.500000,83.333333,796.250000,1895,no,49.961176,6,...,184,0.12,4.430000,5.242000,97.33,92.0,0.03,0.28,0.05,0.16
3,4,25,99.976868,2.916667,76.250000,695.833333,1645,no,49.960900,9,...,197,0.24,4.500000,5.221250,94.50,89.0,0.03,0.30,0.05,0.18
4,5,25,99.968284,2.500000,121.250000,829.166667,1971,no,50.000000,5,...,205,0.19,3.960000,5.233000,92.00,88.0,0.04,0.31,0.05,0.18


In [372]:
# Drop the 6 duplicated columns in Laboratory and Process
process= process.drop(columns=['Drug release average (%)', 'Drug release min (%)', 'Residual solvent',
                      'Total impurities', 'Impurity O', 'Impurity L'])
process.shape

(1005, 29)

In [373]:
# Rename columns
process.columns = process.columns.str.lower().str.replace(' ', '_')
print(process.columns)

Index(['batch', 'code', 'tbl_speed_mean', 'tbl_speed_change',
       'tbl_speed_0_duration', 'total_waste', 'startup_waste', 'weekend',
       'fom_mean', 'fom_change', 'srel_startup_mean', 'srel_production_mean',
       'srel_production_max', 'main_compforce_mean', 'main_compforce_sd',
       'main_compforce_median', 'pre_compforce_mean', 'tbl_fill_mean',
       'tbl_fill_sd', 'cyl_height_mean', 'stiffness_mean', 'stiffness_max',
       'stiffness_min', 'ejection_mean', 'ejection_max', 'ejection_min',
       'startup_tbl_fill_maxdifference', 'startup_main_compforce_mean',
       'startup_tbl_fill_mean'],
      dtype='object')


In [374]:
# Fill 0 values in stiffness and ejection columns with NaN
process[['stiffness_mean', 'stiffness_max', 'ejection_mean', 'ejection_max']] = process[['stiffness_mean', 'stiffness_max',
                                                                                         'ejection_mean', 'ejection_max']].replace(0, np.nan)
round(process[['stiffness_mean', 'stiffness_max', 'ejection_mean', 'ejection_max']].describe(), 2)

,stiffness_mean,stiffness_max,ejection_mean,ejection_max
count,996.00,996.00,1004.00,1004.00
mean,118.87,184.41,188.05,244.83
std,191.36,348.07,111.87,242.05
min,0.01,2.00,0.02,5.00
25%,43.86,58.00,152.24,179.00
50%,64.62,74.00,184.54,220.00
75%,108.72,198.25,216.02,275.25
max,1729.05,2369.00,2253.67,4747.00


In [375]:
# Change weekend to boolean
process['weekend'] = process['weekend'].map({'no': False, 'yes': True})
process['weekend'].value_counts()

,count
weekend,
False,950
True,55


## 4. `Normalization` cleaning

In [376]:
normalization.head()

,Product code,Batch Size (tablets),Normalisation factor
0,1,240000,2.40
1,2,1920000,19.20
2,3,960000,9.60
3,4,583000,5.83
4,5,2400000,24.00


In [377]:
# Rename columns
normalization = normalization.rename(columns={'Product code': 'code',
                                              'Batch Size (tablets)': 'batch_size',
                                              'Normalisation factor': 'norm_factor'})
normalization.head()

,code,batch_size,norm_factor
0,1,240000,2.40
1,2,1920000,19.20
2,3,960000,9.60
3,4,583000,5.83
4,5,2400000,24.00


In [378]:
# Confirm that size (Laboratory) and batch_size (Normalization) now match
check = laboratory[['code', 'size']].drop_duplicates().merge(normalization, on='code')
(check['size'] == check['batch_size']).all()

np.True_

In [379]:
# Drop 'batch_size' on Normalization
normalization = normalization.drop(columns='batch_size')
normalization.head()

,code,norm_factor
0,1,2.40
1,2,19.20
2,3,9.60
3,4,5.83
4,5,24.00


## 5. Missing values

In [380]:
# Check null values in Laboratory
lab_nulls = laboratory.isnull().sum()
lab_nulls[lab_nulls > 0]

,0
api_water,26
api_total_impurities,67
api_l_impurity,370
api_content,2
api_ps01,26
api_ps05,26
api_ps09,26
tbl_min_weight,10
tbl_max_weight,10


In [381]:
# Check null values in Process
proc_nulls = process.isnull().sum()
proc_nulls[proc_nulls > 0]

,0
stiffness_mean,9
stiffness_max,9
ejection_mean,1
ejection_max,1


## 6. Merge tables

In [382]:
# Merge Laboratory and Process on batch
lab_proc = laboratory.merge(process, on=['batch', 'code'], how='left', validate='one_to_one')
lab_proc.shape

(1005, 76)

In [383]:
# Merge the new table to Normalization on code
pharma_final = lab_proc.merge(normalization, on='code', how='left', validate='many_to_one')
pharma_final.shape

(1005, 77)

In [384]:
# Check that every code found its normalization factor
pharma_final['norm_factor'].isna().sum()

np.int64(0)

In [385]:
# Create new column
pharma_final['startup_waste_norm'] = pharma_final['startup_waste']/ pharma_final['norm_factor']
pharma_final.shape

(1005, 78)

In [386]:
# Confirmation
pharma_final.groupby('code')['startup_waste_norm'].mean().round(2).sort_values(ascending=False)

,startup_waste_norm
code,
16,1739.38
9,1655.00
4,1630.94
14,1491.57
8,1249.32
21,1219.84
1,1107.55
25,888.54
23,844.05


## 7. Final verification

In [387]:
# Check the new table info
pharma_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1005 entries, 0 to 1004
Data columns (total 78 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   batch                           1005 non-null   int64         
 1   code                            1005 non-null   int64         
 2   strength                        1005 non-null   int64         
 3   size                            1005 non-null   int64         
 4   start                           1005 non-null   datetime64[ns]
 5   api_code                        1005 non-null   int64         
 6   api_batch                       1005 non-null   int64         
 7   smcc_batch                      1005 non-null   int64         
 8   lactose_batch                   1005 non-null   int64         
 9   starch_batch                    1005 non-null   int64         
 10  api_water                       979 non-null    float64       
 11  api_

In [388]:
# Check the descriptive values
round(pharma_final.describe().T, 2)

,count,mean,min,25%,50%,75%,max,std
batch,1005.0,503.0,1.0,252.0,503.0,754.0,1005.0,290.262812
code,1005.0,15.423881,1.0,13.0,17.0,22.0,25.0,7.103071
strength,1005.0,16.114428,5.0,10.0,20.0,20.0,40.0,10.047291
size,1005.0,1003408.955224,240000.0,583000.0,960000.0,1100000.0,2400000.0,571110.157214
start,1005,2019-12-30 04:19:20.597015040,2018-11-01 00:00:00,2019-06-01 00:00:00,2019-12-01 00:00:00,2020-09-01 00:00:00,2021-04-01 00:00:00,NaN
...,...,...,...,...,...,...,...,...
startup_tbl_fill_maxdifference,1005.0,0.428488,0.0,0.18,0.31,0.5,2.98,0.400823
startup_main_compforce_mean,1005.0,6.147976,2.576471,4.714286,5.8,7.170833,17.65,1.954359
startup_tbl_fill_mean,1005.0,5.923931,3.412857,5.286667,5.876,6.4445,9.75531,1.06356
norm_factor,1005.0,10.03409,2.4,5.83,9.6,11.0,24.0,5.711102


## 8. Save the new table

In [389]:
pharma_final.to_csv(f'{PROCESSED}/pharma_clean.csv', index=False)

## 9. Cleaning summary

| # | Table | Change | Reason |
|---|---|---|---|
| 1 | `Laboratory` | Renamed `resodual_solvent` → `residual_solvent` | Typo in the original file |
| 2 | `Laboratory` | Converted 6 API columns (`api_water`, `api_total_impurities`, `api_l_impurity`, `api_ps01`, `api_ps05`, `api_ps09`) to numeric | Missing values were stored as blank spaces, so the columns were read as text |
| 3 | `Laboratory` | Removed letters from `strength` and converted it to integer (`5MG` → 5) | Inconsistent format (`5MG` vs `20M`); numeric values allow correct sorting |
| 4 | `Laboratory` | Translated Slovenian months (`maj`, `avg`, `okt`) and converted `start` to datetime | Required for time-based analysis (I-MR charts) |
| 5 | `Laboratory` | Dropped the 6 hardness columns | Average hardness falls outside the min–max range in 479 and 337 batches; tensile strength is used instead |
| 6 | `Laboratory` | Fixed `size` for codes 14 and 16 (4,800,000 → 480,000) | Consistent with the normalization factor (4.8) |
| 7 | `Process` | Dropped the 6 duplicated quality columns | Identical to the `Laboratory` quality columns |
| 8 | `Process` | Renamed columns to snake_case | Remove spaces and capital letters |
| 9 | `Process` | Replaced zeros with NaN in `stiffness_mean`, `stiffness_max`, `ejection_mean` and `ejection_max` | A mean or max of 0 means the sensor did not record; zeros in `_min` columns were kept (press stops) |
| 10 | `Process` | Converted `weekend` to boolean | Yes/no variable |
| 11 | `Normalization` | Renamed columns (`code`, `norm_factor`) and dropped `batch_size` | `code` matches the other tables; `batch_size` duplicated `size` after fix #6 |
| 12 | All | Missing values kept as NaN (no imputation, no row removal) | Dropping rows would remove most of API source 3 (48% of batches); imputing 37% of `api_l_impurity` would create artificial data |
| 13 | All | Merged `Laboratory` + `Process` (on `batch`, `code`) + `Normalization` (on `code`) | Single dataset for analysis and Power BI |
| 14 | Merged | Created `startup_waste_norm` = `startup_waste` / `norm_factor` | `startup_waste` is not normalized by batch size (unlike `total_waste`) |

**Notes:**
- Batches follow chronological order, except batch 73 (Nov 2018), which appears after a later month. For time-based analysis, data will be sorted by `start` and then `batch`.
- `api_l_impurity` will only be used with the 635 batches that have values.